# DICE — Notebook 1.1 : notions fondamentales — Version étudiante

Ce notebook propose une introduction minimale au modèle DICE afin d’en simuler les mécanismes étape par étape.

**Programme**
- Charger le module `DICE.py` fourni avec le cours.
- Construire la grille temporelle et la matrice des états initiaux.
- Choisir une trajectoire simple de réduction des émissions $\mu_t$ et simuler le modèle.
- Tracer une variable endogène, par exemple $T_{AT}$, et une variable exogène, par exemple $F_{EX}$.

> L’optimisation avancée n’est pas abordée ici ; elle sera étudiée dans les notebooks suivants.


## 0) Préparation


In [ ]:
# Exécutez cette cellule une fois pour vérifier/installer les paquets Python requis pour ce portable.

import sys
import subprocess
import importlib.util

required = {
    "matplotlib": "matplotlib",
    "numba": "numba",
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "tqdm": "tqdm",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing]
    )

print("Python environment ready.")


In [ ]:
# Importations standard
import sys, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 1) Importer le module DICE et inspecter les paramètres

In [ ]:
from DICE import (
    Params,
    init_states,
    update_path,
    mat_to_df,
    obj_fun,
    run_optimal_policy,
)

# Créer un objet paramètre par défaut
p = Params()
print(p)
print("\nNombre de périodes nT =", p.nT, "avec un pas Δ =", p.Delta, "années")
print("Time runs from", p.t0, "to", p.tT)


## 2) Construire la matrice des états initiaux

### Temps et variables du modèle

Le modèle est écrit en temps discret sur un horizon fini $T$ :
- les périodes sont indexées par $i=0,1,\ldots,I$ ;
- le pas de temps est $\Delta$ années ;
- l’année civile est $t(i)=t_0+i\Delta$.

La grille temporelle est donc
$$
t=\{t_0,t_0+\Delta,t_0+2\Delta,\ldots,T\}.
$$

Une variable peut être notée $Y_t$ en temps civil ou $Y(i)$ avec l’indice discret.

### États exogènes

Le vecteur $x_t\in\mathbb{R}^{N_x}$ contient notamment la productivité $A_t$, l’intensité carbone $\sigma_t$, le forçage non lié au CO₂ $F_t^{EX}$, les émissions d’usage des sols $E_t^{land}$ et la population $L_t$. Il suit
$$
x_t=g(x_{t-1}),\quad g:\mathbb{R}^{N_x}\to\mathbb{R}^{N_x}.
$$

Ces variables sont stockées dans la matrice `y`; les variables encore inconnues restent à `NaN`.


In [ ]:
# Initialiser la matrice de simulation avec les moteurs exogènes et les états initiaux
sim = init_states(p)

# Convertir en un cadre de données bien rangé pour l'inspection (facultatif)
df0 = mat_to_df(sim, p)
df0.head()


## 3) Choisir des contrôles simples et simuler

Le vecteur endogène $y_t\in\mathbb{R}^{N_y}$ contient les stocks de carbone $(M_t^{AT},M_t^{UP},M_t^{LO})$, les températures $(T_t^{AT},T_t^{OC})$, la production $Y_t$, le capital $K_t$, etc. Il évolue selon
$$
y_t=f(y_{t-1},x_{t-1},z_t),\quad f:\mathbb{R}^{N_y}\times\mathbb{R}^{N_x}\times\mathbb{R}^2\to\mathbb{R}^{N_y}.
$$

Les contrôles sont $z_t=[\mu_t,s_t]'$ : le taux de réduction des émissions et le taux d’épargne.

### Résolution récursive en boucle ouverte
Les contrôles étant imposés, le modèle se résout période après période :
1. mettre à jour les états exogènes $x_t=g(x_{t-1})$ ;
2. calculer les états endogènes $y_t=f(y_{t-1},x_{t-1},z_t)$.

On obtient ainsi la trajectoire complète $(x_t,y_t,z_t)$.

### Exemple
Nous n’optimisons pas encore :
- le taux d’épargne $s_t$ est fixé à 20 % ;
- le taux de réduction $\mu_t$ est fixé à 3 % à titre illustratif.


In [ ]:
# Indices de temps à mettre à jour (les tableaux DICE commencent à t=1 pour les transitions)
timevec = range(1, p.nT)

# 3.a) Contrôles: épargne et réduction (illustration — pas optimale)
sim[1:, p.i_s]  = 0.20  # constant saving rate at 20%
# Start abatement at 3%
sim[1:, p.i_mu] =0.03

# 3.b) Propager le système vers l'avant en utilisant la fonction de mise à jour
sim = update_path(sim, timevec, p)

# Afficher quelques lignes pour confirmer les variables mises à jour (p. ex., Y, C, T AT, F EX)
cols_to_show = ['time','s','mu','Y','C','T_AT','F_EX','E_land']
pd.DataFrame(sim, columns=p.col)[cols_to_show].head(8)


## 4) Tracer une variable endogène et un déterminant exogène


In [ ]:
années = sim[:, p.i_time]

# Exemple endogène : température atmosphérique T AT
plt.figure()
plt.plot(années, sim[:, p.i_T_AT], linewidth=2)
plt.xlabel("Année"); plt.ylabel("Atmospheric temperature (°C)")
plt.title("Endogenous path: $T_{AT}$")
plt.grid(True)
plt.show()

# Exemple exogène : force F EX non CO2
plt.figure()
plt.plot(années, sim[:, p.i_F_EX], linewidth=2)
plt.xlabel("Année"); plt.ylabel("Non-CO$_2$ forcing (W/m$^2$)")
plt.title("Exogenous driver: $F_{EX}$")
plt.grid(True)
plt.show()


## 5) Complément : production et consommation


In [ ]:
plt.figure()
plt.plot(années, sim[:, p.i_Q], linewidth=2, label="Production Q")
plt.plot(années, sim[:, p.i_C], linewidth=2, label="Consommation C")
plt.xlabel("Année"); plt.ylabel("Level (model units)")
plt.title("Production et consommation (commandes illustratives)")
plt.grid(True); plt.legend()
plt.show()


## 6) Exercice

Comparez graphiquement :
- un scénario de laisser-faire sans réduction des émissions, $\mu_t=0$ ;
- une transition verte où $\mu_t$ passe linéairement de 0 en 2020 à 1 en 2050.


In [ ]:
# Construisez d'abord le chemin BAU.
sim_bau = init_states(p)
sim_bau[1:, p.i_s] = 0.20
sim_bau[1:, p.i_mu] = 0.0
# Conseil : propagez-le avec update_path(..., range(1, p.nT), p).

# Construire ensuite une nouvelle trajectoire de transition verte.
sim_green = init_states(p)
sim_green[1:, p.i_s] = 0.20
années = sim_green[:, p.i_time]

# Indications :
# 1. Localiser 2020 et 2050 avec np.argmin(np.abs(années - année cible)).
# 2. Créer mu_green = np.zeros(p.nT).
# 3. Remplissez 2020-2050 avec np.linspace(0.0, 1.0, nombre de points).
# 4. Garder mu green égal à 1 après 2050, l'assigner à i_mu,
# propager le chemin, puis comparer mu, émissions et température.


In [ ]:
# Intentionally left as a workspace cell.
